# Qwen3.5-0.8B — ARDB unified page understanding (bbox + text, one forward pass) (Unsloth, LoRA, T4/L4)

Fine-tunes `unsloth/Qwen3.5-0.8B` on `Soxavin/ardb-sft-v5`: one page image in, one JSON list of
regions out, each region carrying its `box_2d`, `label`, and transcribed `text`:

```json
[
  {"box_2d": [15,15,90,90], "label": "Picture", "text": ""},
  {"box_2d": [21,91,51,266], "label": "Page-Furniture", "text": "ធនាគារ ARDB"},
  {"box_2d": [111,15,885,984], "label": "Table", "text": "| ២៣ | ... |"}
]
```

Based directly on Unsloth's official [`Qwen3_5_(0_8B)_Vision.ipynb`](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb)
— only what our task actually needs is changed from it:

- **Install cell**: `torch==2.8.0` → `2.10.0` (the official pin predates a `ScalingType` symbol
  a newer `transformers` needs — confirmed via direct PyTorch source check), and `causal_conv1d`
  gets `--no-binary` added, not just `--no-build-isolation` — without it, a prebuilt wheel built
  for `torch==2.8.0`'s ABI can still get pulled in even when a different torch is installed, and
  loading that mismatched compiled extension is what actually produced a cascade of
  seemingly-unrelated `ImportError`s elsewhere in torch. `transformers` stays at the official
  `5.2.0` — confirmed unnecessary to change once the real cause was fixed. This exact combination
  is confirmed working end-to-end (full training run completed) via a reference notebook the
  mentor provided. `fast-langdetect` is added for the script check below (not part of official).
- **Dataset**: `Soxavin/ardb-sft-v5` instead of `unsloth/LaTeX_OCR`, with a `SMOKE_TEST` toggle
  and an `EPOCHS` sweep knob.
- **`max_length=2048`**, matching official — Unsloth silently clamps anything higher for this
  model regardless of what's requested (`Unsloth: You set max_seq_length as 6144 but the maximum
  the model supports is 2048. We shall reduce it.`, confirmed in the same reference run). Our
  longest Table target runs to ~4000 chars, so the longest pages' targets will be truncated
  during training — a real, model-imposed ceiling to account for when reading eval results, not
  a config choice we can raise our way out of.
- **`fp16`/`bf16` selected via `is_bfloat16_supported()`** — official doesn't need this (T4-only
  demo), but we run on either T4 or L4, which differ on this.
- **Train-only image augmentation** (brightness/contrast jitter + light blur, never on val/test).
- **Base-model Thai-vs-Khmer probes**, before training: an earlier trial
  (`docs/PROJECT_LOG.md` §2.104) found this base model generates Thai instead of Khmer at zero
  training steps. Still worth fine-tuning — mentor-requested, and a documented result either
  way — but these probes re-check the finding against the exact loaded checkpoint rather than
  assume it still holds.
- **A CER/bbox/script eval cell** in place of official's LaTeX-rendering demo, since our target
  is structured JSON, not a single string.

**Steps:** Runtime ▸ Change runtime type ▸ **T4 or L4 GPU** → run all cells with
`SMOKE_TEST = True` first → then `SMOKE_TEST = False` for a full run. Qwen3.5's Gated DeltaNet
kernels can take several extra minutes to compile/build on the first run — expected, not a hang.

In [ ]:
%%capture
import os, importlib.util
# Deviations from the official install cell, in the order they were found necessary (each
# confirmed, not guessed -- full story in the intro cell above):
#   1) torch==2.8.0 -> 2.10.0 -- the official pin predates the `ScalingType` symbol a newer
#      transformers needs (confirmed via direct PyTorch source check).
#   2) causal_conv1d needs --no-binary, not just --no-build-isolation -- without it, uv can
#      still resolve a prebuilt wheel matched to torch==2.8.0's ABI even though we've asked for
#      2.10.0. A mismatched compiled extension loading successfully-but-corrupted is what was
#      actually producing the cascade of seemingly-unrelated ImportErrors elsewhere in torch
#      (ScalingType, is_opentelemetry_available, CUSTOM_KEY) -- not the transformers version,
#      which stays at the official 5.2.0. Confirmed working end-to-end (full training run
#      completed) in a reference notebook the mentor provided with this exact combination.
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.10.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.34 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec>=0.10.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation --no-binary causal_conv1d flash-linear-attention "causal_conv1d>=1.6.0"
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
# fast-langdetect: our own addition (not in the official notebook), used by the eval cell's
# script-detection cross-check.
!pip install -qqq fast-langdetect


In [ ]:
# Outside %%capture on purpose -- prints what actually got installed instead of assuming it.
import torch, unsloth, unsloth_zoo, transformers
print(f"torch={torch.__version__} unsloth={unsloth.__version__} "
     f"unsloth_zoo={unsloth_zoo.__version__} transformers={transformers.__version__}")

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = False, # 16-bit LoRA -- Unsloth's own guidance against 4-bit for this model
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
# Base-model script check, before any training -- re-verifies the §2.104 finding (base model
# generates Thai instead of Khmer) against this exact load. Probe 1: plain text, no image, to
# isolate whether the gap is in the language model itself or specific to the vision path.
import re as _re
_KHMER_RE = _re.compile(r"[ក-៿]")
_THAI_RE = _re.compile(r"[฀-๿]")

def _script_of(text: str) -> str:
    k, t = len(_KHMER_RE.findall(text)), len(_THAI_RE.findall(text))
    if k == 0 and t == 0:
        return "neither"
    return "khmer" if k >= t else f"thai ({t} Thai vs {k} Khmer codepoints)"

FastVisionModel.for_inference(model)
_probe_messages = [{"role": "user", "content": [
    {"type": "text", "text": "Translate this to Khmer: The price of rice today is 4500 riels per kilogram."}]}]
_probe_input = tokenizer.apply_chat_template(_probe_messages, add_generation_prompt=True)
_probe_inputs = tokenizer(None, _probe_input, add_special_tokens=False, return_tensors="pt").to("cuda")
_probe_out = model.generate(**_probe_inputs, max_new_tokens=200, use_cache=True, do_sample=False)
_probe_text = tokenizer.decode(_probe_out[0][_probe_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Probe 1: plain-text Khmer translation ---")
print(_probe_text)
print(f"script: {_script_of(_probe_text)}\n")
FastVisionModel.for_training(model)


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,           # Unsloth's own default for this model -- 0.8B has little capacity to spare
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    # target_modules = "all-linear", # Optional -- can specify a list if needed
)


## Data

One flat schema — `image` + `instruction` + `text` (the JSON list of regions), same repo the
Gemma notebook trains on.

In [ ]:
SMOKE_TEST = True  # flip to False only after a smoke run has completed without errors

# Not yet re-swept for this model/dataset size -- see eval/qwen_finetune_runs.md before
# changing this. Each value pushes its adapter to its own repo, see the push-to-hub cell below.
EPOCHS = 3

from datasets import load_dataset

_REPO_ID = "Soxavin/ardb-sft-v5"
dataset = load_dataset(_REPO_ID, split="train")
val_dataset = load_dataset(_REPO_ID, split="validation")

if SMOKE_TEST:
    dataset = dataset.select(range(min(10, len(dataset))))
    val_dataset = val_dataset.select(range(min(5, len(val_dataset))))

print(f"train rows: {len(dataset)}, validation rows: {len(val_dataset)}")


In [ ]:
import random
from PIL import Image, ImageEnhance, ImageFilter

# Train-only: applied below only on the training conversion path, never on val_dataset. No
# geometric transforms -- box_2d targets would need re-projecting to match, which this doesn't do.
_aug_rng = random.Random(3407)

def augment_image(img: Image.Image, rng: random.Random) -> Image.Image:
    img = ImageEnhance.Brightness(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    img = ImageEnhance.Contrast(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    if rng.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.1, 0.3)))
    return img

In [ ]:
# Probe 2 of 2: the real image+instruction task. LoRA is attached but untrained (an untrained
# adapter is ~0 contribution to the forward pass), so this still measures base-model behavior.
FastVisionModel.for_inference(model)
_probe2_sample = val_dataset[0]
_probe2_messages = [{"role": "user", "content": [
    {"type": "image"}, {"type": "text", "text": _probe2_sample["instruction"]}]}]
_probe2_input = tokenizer.apply_chat_template(_probe2_messages, add_generation_prompt=True)
_probe2_inputs = tokenizer(_probe2_sample["image"], _probe2_input, add_special_tokens=False,
                           return_tensors="pt").to("cuda")
_probe2_out = model.generate(**_probe2_inputs, max_new_tokens=500, use_cache=True, do_sample=False)
_probe2_text = tokenizer.decode(_probe2_out[0][_probe2_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Probe 2: real image+instruction task ---")
print(_probe2_text[:1000])
print(f"script: {_script_of(_probe2_text)}\n")
FastVisionModel.for_training(model)


In [ ]:
def convert_to_conversation(sample, train: bool = False, rng: random.Random | None = None):
    img = augment_image(sample["image"], rng) if train else sample["image"]
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text", "text": sample["instruction"]},
            {"type": "image", "image": img},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]}

converted_dataset = [convert_to_conversation(s, train=True, rng=_aug_rng) for s in dataset]


## Train

No `get_chat_template()` call — the official notebook doesn't call it either; Qwen3.5's default
chat template is already correct for this model.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # -1, never None: TrainingArguments._validate_args unconditionally does
        # `max_steps > 0 and num_train_epochs > 0`, so None crashes the moment it's compared.
        max_steps = 10 if SMOKE_TEST else -1,
        num_train_epochs = 1 if SMOKE_TEST else EPOCHS,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        # 2048, not our usual 6144 -- Unsloth clamps this model to 2048 regardless of what's
        # requested (confirmed in the mentor's reference run), so asking for more is a no-op.
        # See the intro cell for what this means for our longest (~4000-char) Table targets.
        max_length = 2048,
    ),
)


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s used for training.")
print(f"Peak reserved memory = {used_memory} GB / {max_memory} GB.")
print(f"Peak reserved memory for training (LoRA) = {used_memory_for_lora} GB.")


In [ ]:
# Push right after training, before the slower eval cells below -- a Colab disconnect during
# eval would otherwise risk losing the whole run. Requires an HF_TOKEN Colab secret.
from google.colab import userdata

_ADAPTER_REPO_ID = "Soxavin/qwen35-ardb-lora-v5-smoke" if SMOKE_TEST else f"Soxavin/qwen35-ardb-lora-v5-e{EPOCHS}"
_hf_token = userdata.get("HF_TOKEN")
model.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
tokenizer.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
print(f"Pushed to https://huggingface.co/{_ADAPTER_REPO_ID}")


## Inference sanity check

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

sample = val_dataset[0]
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=5000,
                   use_cache=True, temperature=1.0, top_p=0.95, top_k=64)
print("\n--- expected ---\n", sample["text"])


## Evaluate on the validation split (per-label CER + bbox accuracy, not eyeballing)

Each prediction is a JSON list of regions, not one string, so CER against the whole blob isn't
meaningful. Parses both prediction and reference, matches regions by `label`, reports mean CER
per label plus a bbox accuracy signal. Malformed JSON is its own failure count, not a crash.

**Script diagnostic**, given the known base-model risk above: whole-row and per-region-by-label
Unicode codepoint checks, cross-checked with `fast-langdetect` per region (skipped on regions
with no alphabetic content, since its own accuracy guidance says short/non-linguistic text isn't
reliably classifiable).

In [ ]:
import json, re
from fast_langdetect import detect as _langdetect

def levenshtein(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]

def cer(pred: str, ref: str) -> float:
    return levenshtein(pred, ref) / max(1, len(ref))

def parse_regions(text: str) -> list[dict] | None:
    try:
        regions = json.loads(text)
    except json.JSONDecodeError:
        return None
    return regions if isinstance(regions, list) else None

_KHMER_RE = re.compile(r"[ក-៿]")
_THAI_RE = re.compile(r"[฀-๿]")

def detect_script(text: str) -> str:
    khmer_n, thai_n = len(_KHMER_RE.findall(text)), len(_THAI_RE.findall(text))
    if khmer_n == 0 and thai_n == 0:
        return "neither"
    return "khmer" if khmer_n >= thai_n else "thai"

def detect_lang_fast(text: str) -> str:
    stripped = text.strip()
    if not stripped or not any(ch.isalpha() for ch in stripped):
        return "n/a"
    try:
        result = _langdetect(stripped.replace("\n", " "), model="auto", k=1)
        return result[0]["lang"] if result else "n/a"
    except Exception:
        return "n/a"

def generate(sample):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=5000, use_cache=True, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

cer_by_label: dict[str, list[float]] = {}
bbox_diffs: list[float] = []
parse_failures = 0
row_script_counts: dict[str, int] = {"khmer": 0, "thai": 0, "neither": 0}
region_script_counts: dict[str, dict[str, int]] = {}
region_langdetect_counts: dict[str, dict[str, int]] = {}

for i, s in enumerate(val_dataset):
    print(f"[{i + 1}/{len(val_dataset)}] generating doc_id={s['doc_id']} page={s['page']}...")
    pred_text = generate(s)
    row_script_counts[detect_script(pred_text)] += 1
    expected = parse_regions(s["text"]) or []
    predicted = parse_regions(pred_text)
    if predicted is None:
        parse_failures += 1
        continue
    pred_by_label: dict[str, list[dict]] = {}
    for r in predicted:
        pred_by_label.setdefault(r.get("label", ""), []).append(r)
    exp_by_label: dict[str, list[dict]] = {}
    for r in expected:
        exp_by_label.setdefault(r["label"], []).append(r)
    for label, exp_list in exp_by_label.items():
        pred_list = pred_by_label.get(label, [])
        for exp_r, pred_r in zip(exp_list, pred_list):
            pred_region_text = pred_r.get("text", "")
            if label != "Picture" and pred_region_text:
                cer_by_label.setdefault(label, []).append(cer(pred_region_text, exp_r["text"]))
                region_script_counts.setdefault(label, {"khmer": 0, "thai": 0, "neither": 0})
                region_script_counts[label][detect_script(pred_region_text)] += 1
                region_langdetect_counts.setdefault(label, {})
                lang = detect_lang_fast(pred_region_text)
                region_langdetect_counts[label][lang] = region_langdetect_counts[label].get(lang, 0) + 1
            exp_box, pred_box = exp_r.get("box_2d"), pred_r.get("box_2d")
            if exp_box and pred_box and len(exp_box) == 4 and len(pred_box) == 4:
                bbox_diffs.append(sum(abs(a - b) for a, b in zip(exp_box, pred_box)) / 4)

print(f"\nparse failures: {parse_failures} / {len(val_dataset)}")
print(f"script of generated output (whole-row, by codepoint majority): {row_script_counts}")
print("script per region, by label (Unicode codepoint check):")
for label, counts in region_script_counts.items():
    print(f"  {label}: {counts}")
print("language per region, by label (fast-langdetect cross-check; 'n/a' = too short/non-alphabetic to classify):")
for label, counts in region_langdetect_counts.items():
    print(f"  {label}: {counts}")
for label, values in cer_by_label.items():
    print(f"{label}: mean CER = {sum(values) / len(values):.4f} (n={len(values)})")
if bbox_diffs:
    print(f"bbox mean abs diff (0-1000 scale): {sum(bbox_diffs) / len(bbox_diffs):.2f} (n={len(bbox_diffs)})")


## Save

In [ ]:
model.save_pretrained("qwen35_ardb_lora")
tokenizer.save_pretrained("qwen35_ardb_lora")
# model.push_to_hub("your_name/qwen35_ardb_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/qwen35_ardb_lora", token="YOUR_HF_TOKEN")
